In [2]:

import numpy as np
import pandas as pd
from datetime import datetime
import sys
path = "/Users/luke/Desktop/project_intern/result/step3_classifier_full/backtest_combined.parquet"
df = pd.read_parquet(path, engine="pyarrow")

df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= datetime(2026, 1, 26)]
df['pnl_bps'] = df['pnl_bps'].apply(lambda x : x - 0.25)
all_dates = pd.Index(sorted(df["date"].dropna().unique()))

# ---- Daily return: mean pnl_bps of FILLED trades only ----
filled = df[df["valid_trade_flag"] == 1].copy()
print(f"Total trades: {len(df)}, Filled trades: {len(filled)}")
sys.exit()
daily_returns = (
    filled.groupby("date")["pnl_bps"]
    .mean()                          
    .reindex(all_dates, fill_value=0.0)
    .sort_index()
)

num_days = len(daily_returns)
daily_mean = daily_returns.mean()
daily_std = daily_returns.std(ddof=1)
sharpe = np.nan if daily_std == 0 or np.isnan(daily_std) else (daily_mean / daily_std) * np.sqrt(365)

equity = (1.0 + daily_returns / 10_000.0).cumprod()
drawdown = equity / equity.cummax() - 1.0
max_dd = drawdown.min()

total_return = equity.iloc[-1] - 1.0
annual_return = equity.iloc[-1] ** (365 / num_days) - 1.0 if num_days > 0 else np.nan

calmar = np.nan if max_dd == 0 or np.isnan(max_dd) else annual_return / abs(max_dd)

print(f"Days:                {num_days}")
print(f"Filled trades:       {len(filled)}")
print(f"Avg daily return:    {daily_mean:.4f} bps")
print(f"Daily std:           {daily_std:.4f} bps")
print(f"Total return:        {total_return:.6f}")
print(f"Annual return:       {annual_return:.6f}")
print(f"Max drawdown:        {max_dd:.6f}")
print(f"Sharpe ratio:        {sharpe:.4f}")
print(f"Calmar ratio:        {calmar:.4f}")


Total trades: 694413, Filled trades: 282489


SystemExit: 

/Users/luke/Library/Python/3.10/lib/python/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
